# Ứng dụng 3 — Phân tích hành vi và khám phá sở thích khách hàng thương mại điện tử

**Assignment 02 — Phát triển các Hệ thống Thông minh**
Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · GVHD: PGS.TS Trần Đình Quế

Ứng dụng này khác hai ứng dụng trước ở một điểm quyết định: **dữ liệu có văn bản**.
Vì vậy đây là nơi duy nhất trong Assignment 02 phải trình bày đầy đủ chuỗi

$$\text{Bình luận} \rightarrow \text{Token} \rightarrow \text{Token ID}
\rightarrow \text{Vectơ / Embedding}$$

và trả lời bằng số liệu câu hỏi: **văn bản có cải thiện dự đoán so với chỉ dùng
đặc trưng dạng bảng hay không?**

## 1. Định nghĩa bài toán

**Mục tiêu.** Từ thông tin khách hàng và **nội dung đánh giá họ viết**, dự đoán
khách hàng có **khuyến nghị sản phẩm** cho người khác hay không.

- $X$ = đặc trưng hành vi khách hàng $+$ biểu diễn văn bản của bình luận
- $y$ = `Recommended IND` $\in \{0, 1\}$ — $1$ nghĩa là khách hàng khuyến nghị sản phẩm

Đây là bài toán **phân loại nhị phân**.

**Vì sao chọn đúng mục tiêu này.** Đề bài liệt kê nhiều lựa chọn (dự đoán ý định
mua, phân khúc khách hàng, sở thích danh mục…) và yêu cầu **chọn một mục tiêu
được xác định rõ ràng**. `Recommended IND` là lựa chọn đúng ở đây vì ba lý do:

1. Nó **có sẵn nhãn thật** trong dữ liệu — không phải nhãn do ta tự bịa ra bằng
   quy tắc, nên phép đánh giá mới có ý nghĩa.
2. Nó là **tín hiệu thương mại trực tiếp**: khuyến nghị của khách là thứ vận hành
   xếp hạng sản phẩm, gợi ý cá nhân hoá và quyết định nhập hàng.
3. Nó **phụ thuộc mạnh vào nội dung bình luận**, nên là bài toán lý tưởng để đo
   giá trị thật của biểu diễn văn bản — đúng trọng tâm mà đề bài đặt ra.

**Giới hạn phải nói rõ.** Tập dữ liệu **không có `Customer ID`**. Mỗi dòng là một
lượt đánh giá, không phải một khách hàng. Vì vậy mọi kết luận ở đây là **ở cấp
lượt đánh giá**, và ta không thể khẳng định hành vi dài hạn của từng khách hàng.

## 2. Nguồn dữ liệu

| Mục | Giá trị |
|---|---|
| Tên tập dữ liệu | Women's E-Commerce Clothing Reviews |
| Nguồn Kaggle | https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews |
| Số quan sát | 23 486 |
| Số thuộc tính | 11 (10 đặc trưng + 1 biến mục tiêu) |
| Biến mục tiêu | `Recommended IND` (0 = không khuyến nghị, 1 = khuyến nghị) |
| Tệp cục bộ | `data/womens_ecommerce_reviews.csv` |

**Một quan sát là gì.** Mỗi dòng là **một lượt đánh giá do một khách hàng viết cho
một sản phẩm quần áo**, gồm tuổi khách hàng, điểm số 1–5 sao, tiêu đề và nội dung
đánh giá, số lượt người khác thấy hữu ích, và ba cấp phân loại sản phẩm.

In [1]:
# --- 3. Nạp dữ liệu ---
import json
import random
import re
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["font.family"] = "DejaVu Sans"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "womens_ecommerce_reviews.csv"
MODEL_DIR = ROOT / "model"
FIG_DIR = ROOT.parent / "report" / "figures"
MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA, index_col=0)
print("Đã nạp:", DATA)
print("Kích thước (N, cột):", df.shape)
df.head()

Đã nạp: D:\Python\HTTM_Assignment02\Assignment-02-Intelligent-System\customer_behavior\data\womens_ecommerce_reviews.csv
Kích thước (N, cột): (23486, 10)


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


## 4. Khảo sát cấu trúc dữ liệu

In [2]:
print("--- df.shape ---"); print(df.shape)
print("\n--- df.info() ---"); df.info()
print("\n--- df.describe() (cột số) ---")
display(df.describe().T.round(3))

--- df.shape ---
(23486, 10)

--- df.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Clothing ID              23486 non-null  int64
 1   Age                      23486 non-null  int64
 2   Title                    19676 non-null  str  
 3   Review Text              22641 non-null  str  
 4   Rating                   23486 non-null  int64
 5   Recommended IND          23486 non-null  int64
 6   Positive Feedback Count  23486 non-null  int64
 7   Division Name            23472 non-null  str  
 8   Department Name          23472 non-null  str  
 9   Class Name               23472 non-null  str  
dtypes: int64(5), str(5)
memory usage: 9.3 MB

--- df.describe() (cột số) ---


,count,mean,std,min,25%,50%,75%,max
Clothing ID,23486.0,918.119,203.299,0.0,861.0,936.0,1078.0,1205.0
Age,23486.0,43.199,12.280,18.0,34.0,41.0,52.0,99.0
Rating,23486.0,4.196,1.110,1.0,4.0,5.0,5.0,5.0
Recommended IND,23486.0,0.822,0.382,0.0,1.0,1.0,1.0,1.0
Positive Feedback Count,23486.0,2.536,5.702,0.0,0.0,1.0,3.0,122.0


In [3]:
miss = pd.DataFrame({"so_thieu": df.isna().sum()})
miss["ty_le_%"] = (miss["so_thieu"] / len(df) * 100).round(2)
print("--- Giá trị thiếu theo cột ---")
display(miss.sort_values("ty_le_%", ascending=False))
print("Số bản ghi trùng lặp hoàn toàn:", int(df.duplicated().sum()))
print("\n--- Phân bố biến mục tiêu ---")
print(df["Recommended IND"].value_counts())
print(df["Recommended IND"].value_counts(normalize=True).round(4))

--- Giá trị thiếu theo cột ---


,so_thieu,ty_le_%
Title,3810,16.22
Review Text,845,3.60
Class Name,14,0.06
Division Name,14,0.06
Department Name,14,0.06
Clothing ID,0,0.00
Age,0,0.00
Rating,0,0.00
Recommended IND,0,0.00
Positive Feedback Count,0,0.00


Số bản ghi trùng lặp hoàn toàn: 21

--- Phân bố biến mục tiêu ---
Recommended IND
1    19314
0     4172
Name: count, dtype: int64
Recommended IND
1    0.8224
0    0.1776
Name: proportion, dtype: float64


### Phân loại cột theo vai trò

| Nhóm | Cột |
|---|---|
| **Số (numerical)** | `Age`, `Rating`, `Positive Feedback Count`, `Clothing ID` |
| **Phân loại (categorical)** | `Division Name`, `Department Name`, `Class Name` |
| **Văn bản (text)** | `Title`, `Review Text` |
| **Mục tiêu (target)** | `Recommended IND` |

Đây là ứng dụng **duy nhất có đủ cả ba loại**: số, phân loại và văn bản. Chính vì
vậy nó là ứng dụng đòi hỏi biểu diễn phức tạp nhất trong ba ứng dụng.

**Một quyết định quan trọng: `Rating` phải bị loại bỏ.** Xem mục 5.

## 5–9. Phân tích chất lượng dữ liệu

### Vấn đề nghiêm trọng nhất không phải giá trị thiếu — mà là rò rỉ nhãn

`Rating` (số sao 1–5) và `Recommended IND` gần như là **cùng một thông tin**:
khách cho 5 sao thì hầu như chắc chắn khuyến nghị, cho 1 sao thì hầu như chắc
chắn không. Ô dưới đo mức độ trùng khớp ấy.

Nếu giữ `Rating` làm đặc trưng, mọi mô hình sẽ đạt accuracy trên 90% — nhưng
**không mô hình nào học được gì về hành vi khách hàng**, chúng chỉ đọc lại số sao.
Tệ hơn, khi triển khai thật, ta muốn dự đoán khuyến nghị **từ nội dung khách
viết**, mà lúc ấy `Rating` cũng chưa có (khách chưa bấm sao).

Đây là **rò rỉ nhãn (target leakage)** — một dạng rò rỉ dữ liệu tinh vi hơn hẳn
việc `fit` scaler trên tập test, vì nó không lộ ra ở bất kỳ bước kiểm tra pipeline
nào. Điểm số sẽ đẹp, mô hình sẽ vô dụng.

In [4]:
ct = pd.crosstab(df["Rating"], df["Recommended IND"], margins=True)
print("Bảng chéo Rating × Recommended IND:")
display(ct)

agree = pd.crosstab(df["Rating"], df["Recommended IND"], normalize="index").round(4)
print("\nTỷ lệ khuyến nghị theo từng mức sao:")
display(agree)

rule = (df["Rating"] >= 4).astype(int)
acc_rule = (rule == df["Recommended IND"]).mean()
print(f"\nQuy tắc thô 'Rating >= 4 thì khuyến nghị' đạt accuracy = {acc_rule:.4f}")
print("→ Chỉ một câu lệnh if đã đạt trên 90%. Giữ Rating làm đặc trưng là tự lừa mình.")
print("→ QUYẾT ĐỊNH: loại bỏ Rating khỏi tập đặc trưng.")

Bảng chéo Rating × Recommended IND:


Recommended IND,0,1,All
Rating,,,
1,826,16,842
2,1471,94,1565
3,1682,1189,2871
4,168,4909,5077
5,25,13106,13131
All,4172,19314,23486



Tỷ lệ khuyến nghị theo từng mức sao:


Recommended IND,0,1
Rating,,
1,0.9810,0.0190
2,0.9399,0.0601
3,0.5859,0.4141
4,0.0331,0.9669
5,0.0019,0.9981



Quy tắc thô 'Rating >= 4 thì khuyến nghị' đạt accuracy = 0.9365
→ Chỉ một câu lệnh if đã đạt trên 90%. Giữ Rating làm đặc trưng là tự lừa mình.
→ QUYẾT ĐỊNH: loại bỏ Rating khỏi tập đặc trưng.


In [5]:
# --- Trùng lặp và giá trị không hợp lệ ---
print("Trùng lặp hoàn toàn:", int(df.duplicated().sum()))
print("Trùng trên (Review Text, Clothing ID, Rating):",
      int(df.duplicated(subset=["Review Text", "Clothing ID", "Rating"]).sum()))
print("\nBình luận rỗng / thiếu :", int(df["Review Text"].isna().sum()))
print("Tiêu đề thiếu          :", int(df["Title"].isna().sum()))
print("\nMiền giá trị:")
print(f"   Age    : {df['Age'].min()} – {df['Age'].max()}")
print(f"   Rating : {df['Rating'].min()} – {df['Rating'].max()}")
print(f"   Positive Feedback Count: {df['Positive Feedback Count'].min()} – "
      f"{df['Positive Feedback Count'].max()}")

Trùng lặp hoàn toàn: 21
Trùng trên (Review Text, Clothing ID, Rating): 476

Bình luận rỗng / thiếu : 845
Tiêu đề thiếu          : 3810

Miền giá trị:
   Age    : 18 – 99
   Rating : 1 – 5
   Positive Feedback Count: 0 – 122


**Quan sát.** 845 dòng thiếu `Review Text` (3,6%), 3 810 dòng thiếu `Title` (16,2%),
và có bản ghi trùng lặp.

**Diễn giải.** Khách hàng có thể bấm sao mà không viết gì. `Title` là trường tuỳ
chọn nên bị bỏ trống nhiều hơn.

**Ý nghĩa với học máy.** Ứng dụng này lấy văn bản làm trung tâm, nên **dòng không
có bình luận thì không dùng được** — ta loại chúng và nói rõ đã loại bao nhiêu.
`Title` thiếu thì thay bằng chuỗi rỗng rồi ghép vào `Review Text`, vì tiêu đề
thường cô đọng đúng cảm xúc chính ("Love it!", "Disappointed").

In [6]:
# --- Làm sạch ---
before = len(df)
work = df.drop_duplicates(subset=["Review Text", "Clothing ID", "Rating"]).copy()
after_dup = len(work)
work = work[work["Review Text"].notna() & (work["Review Text"].str.strip() != "")]
after = len(work)

print(f"Ban đầu                          : {before:>6} dòng")
print(f"Sau khử trùng lặp                : {after_dup:>6} dòng  (-{before-after_dup})")
print(f"Sau loại dòng không có bình luận : {after:>6} dòng  (-{after_dup-after})")
print(f"Giữ lại {after/before*100:.1f}% dữ liệu gốc")

for c in ["Division Name", "Department Name", "Class Name"]:
    work[c] = work[c].fillna("Unknown").astype(str).str.strip()
work["Title"] = work["Title"].fillna("").astype(str)
work["Review Text"] = work["Review Text"].astype(str)
work["full_text"] = (work["Title"] + " " + work["Review Text"]).str.strip()
print("\nĐã ghép Title + Review Text thành cột 'full_text'.")

Ban đầu                          :  23486 dòng
Sau khử trùng lặp                :  23010 dòng  (-476)
Sau loại dòng không có bình luận :  22639 dòng  (-371)
Giữ lại 96.4% dữ liệu gốc

Đã ghép Title + Review Text thành cột 'full_text'.


## 10. Phân tích khám phá dữ liệu (EDA)

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
cnt = work["Recommended IND"].value_counts().sort_index()
axes[0].bar(["Không khuyến nghị (0)", "Khuyến nghị (1)"], cnt.values, color=["#c0392b", "#27ae60"])
for i, v in enumerate(cnt.values):
    axes[0].text(i, v + 150, f"{v}\n({v/len(work)*100:.1f}%)", ha="center", fontsize=10)
axes[0].set_title("Phân bố biến mục tiêu"); axes[0].set_ylabel("Số lượt đánh giá")

sns.histplot(work["Age"], bins=45, ax=axes[1], color="#8e44ad", kde=True)
axes[1].set_title("Phân bố tuổi khách hàng")
axes[1].set_xlabel("Tuổi"); axes[1].set_ylabel("Số lượt đánh giá")

dep = work["Department Name"].value_counts()
axes[2].bar(dep.index, dep.values, color="#2980b9")
axes[2].set_title("Số lượt đánh giá theo nhóm sản phẩm")
axes[2].set_ylabel(""); plt.setp(axes[2].get_xticklabels(), rotation=25, ha="right")
fig.suptitle("Ứng dụng 3 — Tổng quan dữ liệu", y=1.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "ecom_overview.png")
plt.show()
print(cnt)
print("Tỷ lệ mất cân bằng:", round(cnt[1] / cnt[0], 2), ": 1")

Recommended IND
0     4100
1    18539
Name: count, dtype: int64
Tỷ lệ mất cân bằng: 4.52 : 1


**Quan sát.** Khoảng 82% lượt đánh giá là khuyến nghị, 18% không — tỷ lệ mất cân
bằng khoảng 4,5 : 1, **nặng hơn hẳn** Ứng dụng 1 (1,87 : 1).

**Diễn giải.** Đây là **thiên lệch tự chọn**: người mua hài lòng có xu hướng viết
đánh giá nhiều hơn người thất vọng (người thất vọng thường chỉ trả hàng rồi thôi).

**Ý nghĩa với học máy.** Baseline "luôn đoán khuyến nghị" sẽ đạt tới **82%
accuracy**. Với mức mất cân bằng này, accuracy gần như vô dụng để xếp hạng mô
hình — ta phải dùng **F1 của lớp thiểu số** và ROC-AUC, đồng thời bật
`class_weight="balanced"` ở các mô hình hỗ trợ.

In [8]:
work["text_len"] = work["full_text"].str.len()
work["word_count"] = work["full_text"].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
sns.histplot(work["word_count"], bins=50, ax=axes[0], color="#16a085")
axes[0].set_title("Phân bố độ dài bình luận")
axes[0].set_xlabel("Số từ"); axes[0].set_ylabel("Số lượt đánh giá")
axes[0].axvline(work["word_count"].median(), color="red", ls="--",
                label=f"Trung vị = {work['word_count'].median():.0f} từ")
axes[0].legend(fontsize=9)

sns.boxplot(x="Recommended IND", y="word_count", data=work, ax=axes[1], palette=["#c0392b", "#27ae60"])
axes[1].set_title("Độ dài bình luận theo nhãn")
axes[1].set_xlabel("0 = không khuyến nghị, 1 = khuyến nghị"); axes[1].set_ylabel("Số từ")

rate_by_dep = work.groupby("Department Name")["Recommended IND"].mean().sort_values()
axes[2].barh(rate_by_dep.index, rate_by_dep.values, color="#e67e22")
for i, v in enumerate(rate_by_dep.values):
    axes[2].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
axes[2].set_title("Tỷ lệ khuyến nghị theo nhóm sản phẩm")
axes[2].set_xlabel("Tỷ lệ khuyến nghị"); axes[2].set_xlim(0, 1)
fig.suptitle("Ứng dụng 3 — Đặc điểm bình luận và hành vi", y=1.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "ecom_text_behavior.png")
plt.show()

print("Độ dài bình luận (số từ):")
print(work.groupby("Recommended IND")["word_count"].describe().round(2))

Độ dài bình luận (số từ):
                   count   mean    std  min   25%   50%   75%    max
Recommended IND                                                     
0                 4100.0  65.20  27.57  3.0  42.0  64.0  92.0  118.0
1                18539.0  62.64  29.39  2.0  38.0  61.0  92.0  121.0


**Quan sát.** Bình luận của nhóm **không** khuyến nghị **dài hơn** nhóm khuyến nghị.

**Diễn giải.** Khách hài lòng viết ngắn ("Love this dress!"); khách thất vọng
viết dài để giải thích cái gì sai — sai size, vải mỏng, màu khác ảnh.

**Ý nghĩa với học máy.** Bản thân **độ dài văn bản đã là một đặc trưng dự báo**,
độc lập với nội dung từ ngữ. Ta đưa `word_count` vào nhánh đặc trưng dạng bảng.
Đây là ví dụ đẹp về việc một thuộc tính **siêu dữ liệu** của văn bản mang thông
tin mà mô hình túi-từ không tự nắm được.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

def top_terms(texts, n=18):
    cv = CountVectorizer(stop_words="english", max_features=4000, ngram_range=(1, 1))
    m = cv.fit_transform(texts)
    freq = np.asarray(m.sum(axis=0)).ravel()
    return pd.Series(freq, index=cv.get_feature_names_out()).sort_values(ascending=False).head(n)

pos_terms = top_terms(work.loc[work["Recommended IND"] == 1, "full_text"])
neg_terms = top_terms(work.loc[work["Recommended IND"] == 0, "full_text"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(pos_terms.index[::-1], pos_terms.values[::-1], color="#27ae60")
axes[0].set_title("Từ xuất hiện nhiều nhất — nhóm KHUYẾN NGHỊ")
axes[1].barh(neg_terms.index[::-1], neg_terms.values[::-1], color="#c0392b")
axes[1].set_title("Từ xuất hiện nhiều nhất — nhóm KHÔNG khuyến nghị")
for ax in axes:
    ax.set_xlabel("Tần suất")
fig.suptitle("Ứng dụng 3 — Từ khoá đặc trưng theo nhãn", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "ecom_terms.png")
plt.show()

only_neg = [t for t in neg_terms.index if t not in set(pos_terms.index)]
only_pos = [t for t in pos_terms.index if t not in set(neg_terms.index)]
print("Chỉ xuất hiện trong nhóm KHÔNG khuyến nghị:", only_neg)
print("Chỉ xuất hiện trong nhóm khuyến nghị      :", only_pos)

Chỉ xuất hiện trong nhóm KHÔNG khuyến nghị: ['look', 'really', 'ordered', 'material', 'shirt', 'way', 'quality']
Chỉ xuất hiện trong nhóm khuyến nghị      : ['great', 'perfect', 'beautiful', 'little', 'flattering', 'soft', 'comfortable']


**Quan sát.** Hai nhóm chia sẻ nhiều từ chung (`dress`, `fabric`, `size`,
`fit`) nhưng khác nhau ở một số từ mang cảm xúc và ở từ chỉ vấn đề.

**Diễn giải.** Danh từ sản phẩm xuất hiện ở cả hai nhóm nên **không phân biệt
được**; thứ phân biệt là **tính từ đánh giá** và **từ chỉ lỗi**. Đây chính xác là
lý do phải dùng **TF-IDF chứ không phải đếm tần suất thô**: TF-IDF hạ trọng số
những từ xuất hiện ở khắp nơi và nâng trọng số những từ mang tính phân biệt.

**Ý nghĩa với học máy.** Tín hiệu phân loại nằm trong văn bản, nhưng chỉ ở một
phần nhỏ từ vựng. Mục 18 sẽ đo xem tín hiệu ấy đáng giá bao nhiêu điểm F1.

## 11–12. Biểu diễn dữ liệu

Ứng dụng này cần **hai biểu diễn song song**, rồi ghép lại.

### Nhánh A — đặc trưng dạng bảng

$$x^{\text{bảng}}_i = [\text{Age}, \text{Positive Feedback Count},
\text{word\_count}, \text{Division}, \text{Department}, \text{Class}]$$

Số thì chuẩn hoá, phân loại thì one-hot — giống hệt Ứng dụng 2.

### Nhánh B — biểu diễn văn bản

Đây là phần mà đề bài yêu cầu trình bày đầy đủ:

$$\text{Bình luận} \rightarrow \text{Token} \rightarrow \text{Token ID}
\rightarrow \text{Vectơ} \rightarrow E \in \mathbb{R}^{B \times T \times d}$$

Ô tiếp theo minh hoạ từng bước **trên một bình luận thật**.

In [10]:
sample_text = work["full_text"].iloc[3]
print("=" * 78)
print("BƯỚC 1 — BÌNH LUẬN GỐC (chuỗi ký tự)")
print("=" * 78)
print(f'   "{sample_text}"')
print(f"   Kiểu dữ liệu: {type(sample_text).__name__}, độ dài {len(sample_text)} ký tự")
print("   → Mô hình học máy KHÔNG nhận được chuỗi ký tự.")

print("\n" + "=" * 78)
print("BƯỚC 2 — TÁCH TOKEN (Tokenization)")
print("=" * 78)
tokens = re.findall(r"[a-z0-9']+", sample_text.lower())
print(f"   Số token: {len(tokens)}")
print(f"   20 token đầu: {tokens[:20]}")
print("   → Văn bản đã thành danh sách đơn vị ngôn ngữ, nhưng vẫn là chữ.")

BƯỚC 1 — BÌNH LUẬN GỐC (chuỗi ký tự)
   "My favorite buy! I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!"
   Kiểu dữ liệu: str, độ dài 141 ký tự
   → Mô hình học máy KHÔNG nhận được chuỗi ký tự.

BƯỚC 2 — TÁCH TOKEN (Tokenization)
   Số token: 25
   20 token đầu: ['my', 'favorite', 'buy', 'i', 'love', 'love', 'love', 'this', 'jumpsuit', "it's", 'fun', 'flirty', 'and', 'fabulous', 'every', 'time', 'i', 'wear', 'it', 'i']
   → Văn bản đã thành danh sách đơn vị ngôn ngữ, nhưng vẫn là chữ.


In [11]:
print("=" * 78)
print("BƯỚC 3 — TOKEN ID (ánh xạ mỗi token thành một số nguyên)")
print("=" * 78)
vocab_demo = {t: i for i, t in enumerate(sorted(set(tokens)))}
token_ids = [vocab_demo[t] for t in tokens]
print(f"   Kích thước từ điển minh hoạ: {len(vocab_demo)} token duy nhất")
print("   Ánh xạ 10 token đầu:")
for t in tokens[:10]:
    print(f"      '{t:<12}' → ID {vocab_demo[t]}")
print(f"\n   Chuỗi Token ID (20 đầu): {token_ids[:20]}")
print(f"   Kiểu: danh sách số nguyên, độ dài T = {len(token_ids)}")
print("   → Đã thành số, nhưng ID chỉ là NHÃN, không mang ngữ nghĩa:")
print("     ID 5 không 'lớn hơn' hay 'gần' ID 4 về mặt ý nghĩa.")

BƯỚC 3 — TOKEN ID (ánh xạ mỗi token thành một số nguyên)
   Kích thước từ điển minh hoạ: 21 token duy nhất
   Ánh xạ 10 token đầu:
      'my          ' → ID 16
      'favorite    ' → ID 6
      'buy         ' → ID 2
      'i           ' → ID 11
      'love        ' → ID 15
      'love        ' → ID 15
      'love        ' → ID 15
      'this        ' → ID 18
      'jumpsuit    ' → ID 14
      'it's        ' → ID 13

   Chuỗi Token ID (20 đầu): [16, 6, 2, 11, 15, 15, 15, 18, 14, 13, 8, 7, 0, 5, 4, 19, 11, 20, 12, 11]
   Kiểu: danh sách số nguyên, độ dài T = 25
   → Đã thành số, nhưng ID chỉ là NHÃN, không mang ngữ nghĩa:
     ID 5 không 'lớn hơn' hay 'gần' ID 4 về mặt ý nghĩa.


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 78)
print("BƯỚC 4 — VECTƠ TF-IDF (biểu diễn số có mang trọng số ngữ nghĩa)")
print("=" * 78)
demo_tfidf = TfidfVectorizer(stop_words="english", max_features=200)
demo_tfidf.fit(work["full_text"].head(3000))
vec_demo = demo_tfidf.transform([sample_text])
print(f"   Kích thước từ vựng minh hoạ (d): {len(demo_tfidf.vocabulary_)}")
print(f"   Vectơ kết quả: thưa, hình dạng {vec_demo.shape}, "
      f"{vec_demo.nnz} phần tử khác 0")
nz = vec_demo.toarray().ravel()
names = demo_tfidf.get_feature_names_out()
top_idx = np.argsort(nz)[::-1][:8]
print("   8 chiều có trọng số cao nhất:")
for i in top_idx:
    if nz[i] > 0:
        print(f"      '{names[i]:<14}' → {nz[i]:.4f}")
print("\n   → Mỗi chiều là MỘT TỪ; giá trị là tầm quan trọng của từ đó")
print("     trong bình luận này so với toàn bộ kho văn bản.")

BƯỚC 4 — VECTƠ TF-IDF (biểu diễn số có mang trọng số ngữ nghĩa)


   Kích thước từ vựng minh hoạ (d): 200
   Vectơ kết quả: thưa, hình dạng (1, 200), 8 phần tử khác 0
   8 chiều có trọng số cao nhất:
      'love          ' → 0.5252
      'favorite      ' → 0.3843
      'compliments   ' → 0.3567
      'fun           ' → 0.3510
      'buy           ' → 0.3504
      'time          ' → 0.3462
      'wear          ' → 0.2081
      'great         ' → 0.2003

   → Mỗi chiều là MỘT TỪ; giá trị là tầm quan trọng của từ đó
     trong bình luận này so với toàn bộ kho văn bản.


### Bước 5 — Embedding và tensor $E \in \mathbb{R}^{B \times T \times d}$

TF-IDF cho mỗi văn bản **một vectơ duy nhất** (biểu diễn túi-từ), tức là
$X^{\text{text}} \in \mathbb{R}^{N \times d}$ — mất hoàn toàn thứ tự từ.

Mô hình mạng nơ-ron dùng cách khác: mỗi **token** được ánh xạ thành một vectơ
$d$ chiều, nên một văn bản thành ma trận $T \times d$, và một lô $B$ văn bản
thành **tensor ba chiều**:

$$E \in \mathbb{R}^{B \times T \times d}$$

trong đó $B$ = số văn bản trong lô, $T$ = số token mỗi văn bản (đã đệm về cùng
độ dài), $d$ = số chiều embedding.

Ô dưới **dựng thật tensor ấy** để ba chiều $B$, $T$, $d$ là con số cụ thể chứ
không phải ký hiệu suông.

In [13]:
B, T, D_EMB = 4, 24, 16
batch_texts = work["full_text"].head(B).tolist()

corpus_tokens = [re.findall(r"[a-z0-9']+", t.lower()) for t in work["full_text"].head(6000)]
vocab = {"<PAD>": 0, "<UNK>": 1}
for toks in corpus_tokens:
    for t in toks:
        if t not in vocab:
            vocab[t] = len(vocab)
V = len(vocab)

id_matrix = np.zeros((B, T), dtype=np.int64)          # B × T
for i, text in enumerate(batch_texts):
    toks = re.findall(r"[a-z0-9']+", text.lower())[:T]
    for j, t in enumerate(toks):
        id_matrix[i, j] = vocab.get(t, 1)

rng = np.random.default_rng(RANDOM_SEED)
embedding_table = rng.normal(0, 0.1, size=(V, D_EMB))  # V × d
E = embedding_table[id_matrix]                          # B × T × d

print(f"Kích thước từ điển V = {V:,} token duy nhất")
print(f"\nMa trận Token ID : {id_matrix.shape}  (B × T)")
print(id_matrix[:2, :12], "...")
print(f"\nBảng embedding   : {embedding_table.shape}  (V × d)")
print(f"\nTENSOR EMBEDDING : {E.shape}  (B × T × d)")
print(f"   B = {E.shape[0]:>3}  số văn bản trong lô")
print(f"   T = {E.shape[1]:>3}  số token mỗi văn bản (đã cắt/đệm về cùng độ dài)")
print(f"   d = {E.shape[2]:>3}  số chiều embedding của mỗi token")
print(f"   Tổng số phần tử: {E.size:,} — dtype {E.dtype}")
print(f"\nVectơ embedding của token đầu tiên, văn bản đầu tiên (d = {D_EMB} chiều):")
print("   ", np.round(E[0, 0], 4))

Kích thước từ điển V = 8,513 token duy nhất

Ma trận Token ID : (4, 24)  (B × T)
[[ 2  3  4  5  6  5  7  0  0  0  0  0]
 [ 8  9 10 11 12 13 14 15 16 17 18 19]] ...

Bảng embedding   : (8513, 16)  (V × d)

TENSOR EMBEDDING : (4, 24, 16)  (B × T × d)
   B =   4  số văn bản trong lô
   T =  24  số token mỗi văn bản (đã cắt/đệm về cùng độ dài)
   d =  16  số chiều embedding của mỗi token
   Tổng số phần tử: 1,536 — dtype float64

Vectơ embedding của token đầu tiên, văn bản đầu tiên (d = 16 chiều):
    [-0.0512 -0.0814  0.0616  0.1129 -0.0114 -0.084  -0.0824  0.0651  0.0743
  0.0543 -0.0666  0.0232  0.0117  0.0219  0.0871  0.0224]


**So sánh hai cách biểu diễn văn bản.**

| | TF-IDF (dùng để triển khai) | Embedding (minh hoạ khái niệm) |
|---|---|---|
| Hình dạng | $\mathbb{R}^{N \times d}$ — hai chiều | $\mathbb{R}^{B \times T \times d}$ — ba chiều |
| Giữ thứ tự từ | Không | Có |
| Mỗi chiều nghĩa là gì | Một từ cụ thể — **đọc được** | Chiều ẩn học được — không đọc được |
| Chi phí huấn luyện | Giây | Giờ, cần GPU |
| Cần bao nhiêu dữ liệu | Vài nghìn dòng là đủ | Hàng trăm nghìn dòng trở lên |

**Ta chọn TF-IDF để triển khai.** Với 22 000 dòng, embedding học từ đầu sẽ quá
khớp; hơn nữa TF-IDF cho phép **chỉ ra từ nào đẩy dự đoán** — điều mà mục 20 sẽ
khai thác và là thứ một hệ thống thương mại điện tử cần để giải thích kết quả cho
người vận hành. Tensor $B \times T \times d$ ở trên vì vậy đóng vai trò minh hoạ
khái niệm mà Bài giảng 02 yêu cầu, không phải mô hình được triển khai.

**Thông tin nào bị mất, thông tin nào được giữ.**
Giữ: từ nào xuất hiện, mức độ hiếm của từ đó, cường độ tín hiệu cảm xúc.
Mất: thứ tự từ, phủ định tầm xa ("not good" bị tách thành hai token độc lập —
một phần được bù bằng n-gram bậc 2), mỉa mai, ngữ cảnh vượt câu.

## 13–14. Kỹ thuật đặc trưng và chia tập dữ liệu

In [14]:
from sklearn.model_selection import train_test_split

TEXT_COL = "full_text"
NUM_FEATURES = ["Age", "Positive Feedback Count", "word_count"]
CAT_FEATURES = ["Division Name", "Department Name", "Class Name"]
TAB_FEATURES = NUM_FEATURES + CAT_FEATURES
# Rating bị loại có chủ đích — xem mục 5 (rò rỉ nhãn).

X = work[TAB_FEATURES + [TEXT_COL]].copy()
y = work["Recommended IND"].copy()

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=RANDOM_SEED, stratify=y_tmp)

display(pd.DataFrame({
    "Tập": ["Train", "Validation", "Test", "Tổng"],
    "Số mẫu": [len(X_train), len(X_val), len(X_test), len(X)],
    "Tỷ lệ %": [round(len(t)/len(X)*100, 1) for t in (X_train, X_val, X_test)] + [100.0],
    "Tỷ lệ khuyến nghị %": [round(t.mean()*100, 2) for t in (y_train, y_val, y_test)]
                            + [round(y.mean()*100, 2)],
}))
print("Đặc trưng dạng bảng:", TAB_FEATURES)
print("Cột văn bản        :", TEXT_COL)
print("ĐÃ LOẠI            : Rating (rò rỉ nhãn), Clothing ID (định danh, không khái quát)")

,Tập,Số mẫu,Tỷ lệ %,Tỷ lệ khuyến nghị %
0,Train,15846,70.0,81.89
1,Validation,3397,15.0,81.90
2,Test,3396,15.0,81.89
3,Tổng,22639,100.0,81.89


Đặc trưng dạng bảng: ['Age', 'Positive Feedback Count', 'word_count', 'Division Name', 'Department Name', 'Class Name']
Cột văn bản        : full_text
ĐÃ LOẠI            : Rating (rò rỉ nhãn), Clothing ID (định danh, không khái quát)


## 15. Pipeline tiền xử lý — hai nhánh ghép song song

`ColumnTransformer` cho phép ghép nhánh bảng và nhánh văn bản thành **một
artifact duy nhất**:

| Nhánh | Đầu vào | Các bước | Số chiều đầu ra |
|---|---|---|---|
| Số | 3 cột | `SimpleImputer(median)` → `StandardScaler` | 3 |
| Phân loại | 3 cột | `SimpleImputer(constant)` → `OneHotEncoder` | ~30 |
| Văn bản | 1 cột | `TfidfVectorizer(1–2 gram, 20 000 đặc trưng)` | ~20 000 |

Ba biểu diễn này được ghép ngang thành một ma trận thưa duy nhất. Toàn bộ đều
`fit` **chỉ trên tập train** — kể cả từ điển TF-IDF, vì từ điển học từ toàn bộ dữ
liệu cũng là một dạng rò rỉ (mô hình sẽ "biết" những từ chỉ xuất hiện trong test).

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_branch = Pipeline([("impute", SimpleImputer(strategy="median")),
                           ("scale", StandardScaler())])
categorical_branch = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
text_branch = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                              max_features=20000, min_df=2, sublinear_tf=True)

# Biểu diễn 1 — CHỈ đặc trưng dạng bảng
pre_tabular = ColumnTransformer([
    ("num", numeric_branch, NUM_FEATURES),
    ("cat", categorical_branch, CAT_FEATURES),
], remainder="drop")

# Biểu diễn 2 — bảng + văn bản
pre_hybrid = ColumnTransformer([
    ("num", numeric_branch, NUM_FEATURES),
    ("cat", categorical_branch, CAT_FEATURES),
    ("text", text_branch, TEXT_COL),
], remainder="drop")

Xtr_tab = pre_tabular.fit_transform(X_train)
Xva_tab = pre_tabular.transform(X_val)
Xte_tab = pre_tabular.transform(X_test)

Xtr_hyb = pre_hybrid.fit_transform(X_train)
Xva_hyb = pre_hybrid.transform(X_val)
Xte_hyb = pre_hybrid.transform(X_test)

print("BIỂU DIỄN 1 — chỉ đặc trưng dạng bảng")
print(f"   Xtr: {Xtr_tab.shape}  |  Xte: {Xte_tab.shape}")
print("\nBIỂU DIỄN 2 — bảng + văn bản (TF-IDF)")
print(f"   Xtr: {Xtr_hyb.shape}  |  Xte: {Xte_hyb.shape}")
print(f"   → văn bản đóng góp thêm {Xtr_hyb.shape[1] - Xtr_tab.shape[1]:,} chiều")
print(f"   Kiểu ma trận: {type(Xtr_hyb).__name__}")
if hasattr(Xtr_hyb, "nnz"):
    density = Xtr_hyb.nnz / (Xtr_hyb.shape[0] * Xtr_hyb.shape[1])
    print(f"   Mật độ khác 0: {density*100:.4f}%  → ma trận cực thưa, phải lưu ở dạng sparse")

BIỂU DIỄN 1 — chỉ đặc trưng dạng bảng
   Xtr: (15846, 34)  |  Xte: (3396, 34)

BIỂU DIỄN 2 — bảng + văn bản (TF-IDF)
   Xtr: (15846, 20034)  |  Xte: (3396, 20034)
   → văn bản đóng góp thêm 20,000 chiều
   Kiểu ma trận: csr_matrix
   Mật độ khác 0: 0.2095%  → ma trận cực thưa, phải lưu ở dạng sparse


## 16. Mô hình cơ sở (baseline)

In [16]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)

dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
dummy.fit(Xtr_tab, y_train)
yp = dummy.predict(Xte_tab)
BASE_ACC = accuracy_score(y_test, yp)
print("BASELINE — luôn đoán 'khuyến nghị'")
print(f"   Accuracy = {BASE_ACC:.4f}   ← mốc rất cao do mất cân bằng 4,5 : 1")
print(f"   F1 lớp 0 = {f1_score(y_test, yp, pos_label=0, zero_division=0):.4f}"
      "   ← không bắt được ca 'không khuyến nghị' nào")
print("\n→ Accuracy gần như vô dụng ở bài toán này. Xếp hạng bằng F1 và ROC-AUC.")

BASELINE — luôn đoán 'khuyến nghị'
   Accuracy = 0.8189   ← mốc rất cao do mất cân bằng 4,5 : 1
   F1 lớp 0 = 0.0000   ← không bắt được ca 'không khuyến nghị' nào

→ Accuracy gần như vô dụng ở bài toán này. Xếp hạng bằng F1 và ROC-AUC.


## 17. Huấn luyện mô hình

Đề bài yêu cầu **so sánh sáu mô hình** cho ứng dụng thương mại điện tử. Ta thiết
kế bộ sáu mô hình sao cho chúng trả lời được câu hỏi trung tâm — văn bản có giá
trị bao nhiêu:

| # | Mô hình | Biểu diễn | Vai trò |
|---|---|---|---|
| 1 | Logistic Regression | Chỉ bảng | Chuẩn tham chiếu không dùng văn bản |
| 2 | Decision Tree | Chỉ bảng | Kiểm tra xem phi tuyến trên đặc trưng bảng có cứu vãn được không |
| 3 | Random Forest | Chỉ bảng | Trần hiệu năng của biểu diễn chỉ-bảng |
| 4 | Logistic Regression | Bảng + văn bản | Phân loại tuyến tính trên văn bản — mô hình đề bài chỉ đích danh |
| 5 | Linear SVM | Bảng + văn bản | Biên lớn; chuẩn mạnh cho dữ liệu văn bản thưa nhiều chiều |
| 6 | Multinomial Naive Bayes | Chỉ văn bản | Mô hình xác suất cổ điển cho phân loại văn bản |

Ba mô hình đầu và ba mô hình sau dùng **cùng một cách chia dữ liệu**, nên chênh
lệch điểm số giữa hai nhóm **đo đúng đóng góp của biểu diễn văn bản** chứ không
lẫn yếu tố nào khác.

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

# Nhánh chỉ-văn-bản cho Naive Bayes (TF-IDF không âm)
text_only = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                            max_features=20000, min_df=2, sublinear_tf=True)
Xtr_txt = text_only.fit_transform(X_train[TEXT_COL])
Xva_txt = text_only.transform(X_val[TEXT_COL])
Xte_txt = text_only.transform(X_test[TEXT_COL])
print("Biểu diễn chỉ-văn-bản:", Xtr_txt.shape)

SPECS = {
    "logistic_regression_tabular": {
        "label": "Logistic Regression (chỉ bảng)", "repr": "tabular",
        "model": LogisticRegression(max_iter=2000, class_weight="balanced",
                                    random_state=RANDOM_SEED)},
    "decision_tree_tabular": {
        "label": "Decision Tree (chỉ bảng)", "repr": "tabular",
        "model": DecisionTreeClassifier(max_depth=10, min_samples_leaf=20,
                                        class_weight="balanced", random_state=RANDOM_SEED)},
    "random_forest_tabular": {
        "label": "Random Forest (chỉ bảng)", "repr": "tabular",
        "model": RandomForestClassifier(n_estimators=250, max_depth=16, min_samples_leaf=5,
                                        class_weight="balanced", n_jobs=-1,
                                        random_state=RANDOM_SEED)},
    "logistic_regression_hybrid": {
        "label": "Logistic Regression (bảng + văn bản)", "repr": "hybrid",
        "model": LogisticRegression(max_iter=3000, C=4.0, class_weight="balanced",
                                    random_state=RANDOM_SEED)},
    "linear_svm_hybrid": {
        "label": "Linear SVM (bảng + văn bản)", "repr": "hybrid",
        "model": LinearSVC(C=0.5, class_weight="balanced", max_iter=5000,
                           random_state=RANDOM_SEED)},
    "multinomial_nb_text": {
        "label": "Multinomial Naive Bayes (chỉ văn bản)", "repr": "text",
        "model": MultinomialNB(alpha=0.3)},
}

MATRICES = {
    "tabular": (Xtr_tab, Xva_tab, Xte_tab),
    "hybrid": (Xtr_hyb, Xva_hyb, Xte_hyb),
    "text": (Xtr_txt, Xva_txt, Xte_txt),
}

trained, train_times = {}, {}
for key, spec in SPECS.items():
    Xa, _, _ = MATRICES[spec["repr"]]
    t0 = time.perf_counter()
    spec["model"].fit(Xa, y_train)
    train_times[key] = time.perf_counter() - t0
    trained[key] = spec["model"]
    print(f"✓ {spec['label']:<42} {train_times[key]:>7.2f}s  "
          f"[{spec['repr']}, {Xa.shape[1]:,} chiều]")

Biểu diễn chỉ-văn-bản: (15846, 20000)
✓ Logistic Regression (chỉ bảng)                0.10s  [tabular, 34 chiều]
✓ Decision Tree (chỉ bảng)                      0.06s  [tabular, 34 chiều]


✓ Random Forest (chỉ bảng)                      0.94s  [tabular, 34 chiều]


✓ Logistic Regression (bảng + văn bản)          1.78s  [hybrid, 20,034 chiều]


✓ Linear SVM (bảng + văn bản)                   3.51s  [hybrid, 20,034 chiều]
✓ Multinomial Naive Bayes (chỉ văn bản)         0.01s  [text, 20,000 chiều]


## 18–19. Đánh giá và so sánh mô hình

In [18]:
def score(model, Xs, ys):
    pred = model.predict(Xs)
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(Xs)[:, 1]
    elif hasattr(model, "decision_function"):
        proba = model.decision_function(Xs)
    else:
        proba = None
    return {
        "Accuracy": accuracy_score(ys, pred),
        "Precision": precision_score(ys, pred, zero_division=0),
        "Recall": recall_score(ys, pred, zero_division=0),
        "F1": f1_score(ys, pred, zero_division=0),
        "F1 (lớp 0)": f1_score(ys, pred, pos_label=0, zero_division=0),
        "ROC-AUC": roc_auc_score(ys, proba) if proba is not None else np.nan,
    }


rows = []
for key, spec in SPECS.items():
    _, Xv, _ = MATRICES[spec["repr"]]
    s = score(trained[key], Xv, y_val)
    s["Mô hình"] = spec["label"]
    rows.append(s)
val_tbl = pd.DataFrame(rows).set_index("Mô hình").round(4)
print("KẾT QUẢ TRÊN TẬP VALIDATION")
display(val_tbl.sort_values("ROC-AUC", ascending=False))

KẾT QUẢ TRÊN TẬP VALIDATION


,Accuracy,Precision,Recall,F1,F1 (lớp 0),ROC-AUC
Mô hình,,,,,,
Multinomial Naive Bayes (chỉ văn bản),0.8934,0.8980,0.9813,0.9378,0.6276,0.9525
Logistic Regression (bảng + văn bản),0.9082,0.9601,0.9263,0.9429,0.7651,0.9504
Linear SVM (bảng + văn bản),0.9055,0.9552,0.9281,0.9415,0.7548,0.9479
Decision Tree (chỉ bảng),0.5258,0.8462,0.5144,0.6398,0.3059,0.5575
Random Forest (chỉ bảng),0.6111,0.8328,0.6571,0.7346,0.2730,0.5565
Logistic Regression (chỉ bảng),0.5196,0.8374,0.5129,0.6362,0.2929,0.5449


In [19]:
rows = []
for key, spec in SPECS.items():
    _, _, Xt = MATRICES[spec["repr"]]
    s = score(trained[key], Xt, y_test)
    s["Mô hình"] = spec["label"]
    s["Biểu diễn"] = {"tabular": "Chỉ bảng", "hybrid": "Bảng + văn bản",
                      "text": "Chỉ văn bản"}[spec["repr"]]
    rows.append(s)
test_tbl = pd.DataFrame(rows).set_index("Mô hình")
test_tbl = test_tbl[["Biểu diễn", "Accuracy", "Precision", "Recall", "F1",
                     "F1 (lớp 0)", "ROC-AUC"]]
test_tbl.iloc[:, 1:] = test_tbl.iloc[:, 1:].astype(float).round(4)
test_tbl = test_tbl.sort_values("ROC-AUC", ascending=False)
print("KẾT QUẢ TRÊN TẬP TEST — bảng so sánh sáu mô hình")
display(test_tbl)
print(f"\nBaseline accuracy = {BASE_ACC:.4f}")

KẾT QUẢ TRÊN TẬP TEST — bảng so sánh sáu mô hình


,Biểu diễn,Accuracy,Precision,Recall,F1,F1 (lớp 0),ROC-AUC
Mô hình,,,,,,,
Logistic Regression (bảng + văn bản),Bảng + văn bản,0.8958,0.9530,0.9180,0.9352,0.7342,0.9431
Linear SVM (bảng + văn bản),Bảng + văn bản,0.8955,0.9509,0.9198,0.9351,0.7313,0.9414
Multinomial Naive Bayes (chỉ văn bản),Chỉ văn bản,0.8772,0.8855,0.9763,0.9287,0.5587,0.9346
Logistic Regression (chỉ bảng),Chỉ bảng,0.5389,0.8415,0.5383,0.6566,0.2984,0.5519
Decision Tree (chỉ bảng),Chỉ bảng,0.5197,0.8320,0.5182,0.6386,0.2843,0.5389
Random Forest (chỉ bảng),Chỉ bảng,0.6066,0.8244,0.6602,0.7332,0.2511,0.5304



Baseline accuracy = 0.8189


### Câu hỏi trung tâm: văn bản có cải thiện dự đoán không?

Đề bài yêu cầu **so sánh biểu diễn chỉ-bảng với biểu diễn có kèm bình luận**.
Ô dưới tính chênh lệch trực tiếp giữa hai nhóm.

In [20]:
tab_best = test_tbl[test_tbl["Biểu diễn"] == "Chỉ bảng"]
txt_best = test_tbl[test_tbl["Biểu diễn"] != "Chỉ bảng"]

summary = pd.DataFrame({
    "Chỉ đặc trưng bảng (tốt nhất)": [
        tab_best["ROC-AUC"].max(), tab_best["F1"].max(), tab_best["F1 (lớp 0)"].max(),
        tab_best["Accuracy"].max()],
    "Có kèm văn bản (tốt nhất)": [
        txt_best["ROC-AUC"].max(), txt_best["F1"].max(), txt_best["F1 (lớp 0)"].max(),
        txt_best["Accuracy"].max()],
}, index=["ROC-AUC", "F1 (lớp 1)", "F1 (lớp 0)", "Accuracy"]).round(4)
summary["Cải thiện tuyệt đối"] = (summary.iloc[:, 1] - summary.iloc[:, 0]).round(4)
summary["Cải thiện tương đối %"] = ((summary.iloc[:, 1] / summary.iloc[:, 0] - 1) * 100).round(2)
print("TÁC ĐỘNG CỦA BIỂU DIỄN VĂN BẢN")
display(summary)
TEXT_GAIN = summary.loc["ROC-AUC", "Cải thiện tuyệt đối"]
print(f"\n→ Biểu diễn văn bản cải thiện ROC-AUC thêm {TEXT_GAIN:+.4f}")
print(f"→ F1 của lớp thiểu số (không khuyến nghị) cải thiện "
      f"{summary.loc['F1 (lớp 0)','Cải thiện tương đối %']:+.1f}%")

TÁC ĐỘNG CỦA BIỂU DIỄN VĂN BẢN


,Chỉ đặc trưng bảng (tốt nhất),Có kèm văn bản (tốt nhất),Cải thiện tuyệt đối,Cải thiện tương đối %
ROC-AUC,0.5519,0.9431,0.3912,70.88
F1 (lớp 1),0.7332,0.9352,0.2020,27.55
F1 (lớp 0),0.2984,0.7342,0.4358,146.05
Accuracy,0.6066,0.8958,0.2892,47.68



→ Biểu diễn văn bản cải thiện ROC-AUC thêm +0.3912
→ F1 của lớp thiểu số (không khuyến nghị) cải thiện +146.1%


In [21]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = {"Chỉ bảng": "#95a5a6", "Bảng + văn bản": "#2980b9", "Chỉ văn bản": "#16a085"}
t = test_tbl.reset_index()
bar_colors = [colors[b] for b in t["Biểu diễn"]]

axes[0].barh(t["Mô hình"], t["ROC-AUC"], color=bar_colors)
for i, v in enumerate(t["ROC-AUC"]):
    axes[0].text(v + 0.004, i, f"{v:.4f}", va="center", fontsize=9)
axes[0].set_xlabel("ROC-AUC"); axes[0].set_xlim(0.5, 1.0)
axes[0].set_title("ROC-AUC theo mô hình và biểu diễn")

axes[1].barh(t["Mô hình"], t["F1 (lớp 0)"], color=bar_colors)
for i, v in enumerate(t["F1 (lớp 0)"]):
    axes[1].text(v + 0.006, i, f"{v:.4f}", va="center", fontsize=9)
axes[1].set_xlabel("F1 của lớp 'không khuyến nghị'")
axes[1].set_title("Khả năng bắt lớp thiểu số")
axes[1].set_yticklabels([])

from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color=c, label=l) for l, c in colors.items()],
               fontsize=8, loc="lower right")
fig.suptitle("Ứng dụng 3 — Biểu diễn văn bản đóng góp bao nhiêu?", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "ecom_model_comparison.png")
plt.show()

**Quan sát.** Mọi mô hình có dùng văn bản đều vượt xa mọi mô hình chỉ dùng đặc
trưng bảng, và khoảng cách lớn nhất nằm ở **F1 của lớp thiểu số**.

**Diễn giải.** Đặc trưng bảng (tuổi, nhóm hàng, số lượt hữu ích) gần như **không
mang tín hiệu** về việc khách có hài lòng hay không — chúng mô tả *ai đánh giá*
và *đánh giá cái gì*, chứ không mô tả *khách nghĩ gì*. Ý kiến nằm trong văn bản,
và chỉ nằm ở đó.

**Ý nghĩa với học máy.** Đây là câu trả lời trực tiếp cho câu hỏi của đề bài, và
là bằng chứng mạnh nhất trong cả ba ứng dụng cho luận điểm trung tâm của
Assignment 02: **chọn biểu diễn đúng có tác động lớn hơn hẳn chọn thuật toán
mạnh**. Random Forest với 250 cây trên đặc trưng bảng vẫn thua Naive Bayes — mô
hình đơn giản nhất trong sáu — khi Naive Bayes được đọc văn bản.

In [22]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix

best_label = test_tbl.index[0]
best_id = [k for k, s in SPECS.items() if s["label"] == best_label][0]
best_model = trained[best_id]
_, _, Xte_best = MATRICES[SPECS[best_id]["repr"]]

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, (key, spec) in zip(axes.ravel(), SPECS.items()):
    _, _, Xt = MATRICES[spec["repr"]]
    cm = confusion_matrix(y_test, trained[key].predict(Xt))
    ConfusionMatrixDisplay(cm, display_labels=["Không KN", "Khuyến nghị"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(spec["label"], fontsize=9)
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thực tế")
fig.suptitle("Ứng dụng 3 — Ma trận nhầm lẫn sáu mô hình trên tập test", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "ecom_confusion.png")
plt.show()

cm = confusion_matrix(y_test, best_model.predict(Xte_best))
tn, fp, fn, tp = cm.ravel()
print(f"Diễn giải ma trận nhầm lẫn — {best_label}\n")
print(f"  TP = {tp:>4}  khách khuyến nghị, dự đoán đúng")
print(f"  TN = {tn:>4}  khách KHÔNG khuyến nghị, phát hiện đúng → tín hiệu cảnh báo sản phẩm")
print(f"  FP = {fp:>4}  khách không khuyến nghị nhưng bị đoán là hài lòng")
print(f"       → nguy hiểm về mặt kinh doanh: phản hồi xấu bị bỏ lọt, sản phẩm lỗi tiếp tục bán")
print(f"  FN = {fn:>4}  khách hài lòng bị đoán là không hài lòng")
print(f"       → chỉ gây rà soát thừa, chi phí thấp")
print(f"\n  → Với bài toán này, FP đắt hơn FN, nên Recall của LỚP 0 mới là chỉ số vận hành.")
print(f"     Recall lớp 0 = TN/(TN+FP) = {tn}/{tn+fp} = {tn/(tn+fp):.4f}")

Diễn giải ma trận nhầm lẫn — Logistic Regression (bảng + văn bản)

  TP = 2553  khách khuyến nghị, dự đoán đúng
  TN =  489  khách KHÔNG khuyến nghị, phát hiện đúng → tín hiệu cảnh báo sản phẩm
  FP =  126  khách không khuyến nghị nhưng bị đoán là hài lòng
       → nguy hiểm về mặt kinh doanh: phản hồi xấu bị bỏ lọt, sản phẩm lỗi tiếp tục bán
  FN =  228  khách hài lòng bị đoán là không hài lòng
       → chỉ gây rà soát thừa, chi phí thấp

  → Với bài toán này, FP đắt hơn FN, nên Recall của LỚP 0 mới là chỉ số vận hành.
     Recall lớp 0 = TN/(TN+FP) = 489/615 = 0.7951


## 20. Phân tích sai số và khám phá sở thích khách hàng

In [23]:
# Từ nào đẩy dự đoán về phía nào — chỉ mô hình tuyến tính mới đọc được hệ số.
lr = trained["logistic_regression_hybrid"]
names_hyb = pre_hybrid.get_feature_names_out()
coefs = lr.coef_.ravel()
text_mask = np.array([n.startswith("text__") for n in names_hyb])
text_names = np.array([n.replace("text__", "") for n in names_hyb])[text_mask]
text_coefs = coefs[text_mask]

order = np.argsort(text_coefs)
neg_terms = pd.Series(text_coefs[order[:18]], index=text_names[order[:18]])
pos_terms = pd.Series(text_coefs[order[-18:]], index=text_names[order[-18:]])

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.5))
axes[0].barh(pos_terms.index, pos_terms.values, color="#27ae60")
axes[0].set_title("Cụm từ đẩy MẠNH NHẤT về phía 'khuyến nghị'")
axes[0].set_xlabel("Hệ số hồi quy")
axes[1].barh(neg_terms.index, neg_terms.values, color="#c0392b")
axes[1].set_title("Cụm từ đẩy MẠNH NHẤT về phía 'không khuyến nghị'")
axes[1].set_xlabel("Hệ số hồi quy")
fig.suptitle("Ứng dụng 3 — Khám phá sở thích khách hàng từ hệ số mô hình", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "ecom_coefficients.png")
plt.show()

print("Cụm từ tiêu cực mạnh nhất (dấu hiệu vấn đề sản phẩm):")
print(neg_terms.sort_values().head(12).round(3))

Cụm từ tiêu cực mạnh nhất (dấu hiệu vấn đề sản phẩm):
disappointed    -10.014
wanted love      -9.339
unflattering     -6.711
returning        -6.584
huge             -6.398
disappointing    -6.262
meh              -6.001
bad              -5.937
poor             -5.931
returned         -5.826
cheap            -5.414
return           -5.160
dtype: float64


**Quan sát.** Các cụm từ tiêu cực mạnh nhất chỉ vào **vấn đề cụ thể của sản
phẩm**: sai kích cỡ, chất liệu mỏng, khác ảnh, phải trả hàng. Các cụm từ tích cực
chỉ vào cảm xúc và độ vừa vặn.

**Diễn giải — đây chính là "khám phá sở thích khách hàng" mà đề bài yêu cầu.**
Mô hình không chỉ dự đoán một nhãn; hệ số của nó là **một danh sách xếp hạng
những điều khách hàng quan tâm nhất**, rút tự động từ 22 000 bình luận mà không
cần ai đọc tay.

**Ứng dụng thương mại.** Doanh nghiệp dùng được ngay ba việc: (1) cảnh báo sớm
sản phẩm có vấn đề khi cụm từ tiêu cực tăng đột biến; (2) sửa bảng thông số kích
cỡ ở đúng những mã hàng bị than phiền về size; (3) ưu tiên hiển thị sản phẩm được
khen đúng những thuộc tính khách coi trọng.

In [24]:
proba_best = (best_model.predict_proba(Xte_best)[:, 1]
              if hasattr(best_model, "predict_proba")
              else best_model.decision_function(Xte_best))
pred_best = best_model.predict(Xte_best)

err = pd.DataFrame({
    "van_ban": X_test[TEXT_COL].to_numpy(),
    "y_that": y_test.to_numpy(),
    "y_du_doan": pred_best,
    "diem": np.round(proba_best, 3),
})
wrong = err[err.y_that != err.y_du_doan]
print(f"Số dự đoán sai: {len(wrong)} / {len(err)} ({len(wrong)/len(err)*100:.2f}%)")
print("\n4 bình luận bị phân loại sai (rút gọn 220 ký tự):\n")
for _, r in wrong.head(4).iterrows():
    print(f"[thật={r.y_that} dự đoán={r.y_du_doan} điểm={r.diem}]")
    print(f"   {r.van_ban[:220]}...\n")

Số dự đoán sai: 354 / 3396 (10.42%)

4 bình luận bị phân loại sai (rút gọn 220 ký tự):

[thật=0 dự đoán=1 điểm=0.995]
   Love it! I didn't realize how much i would love this. wore it to work the other day and got more compliments than i have in a very long time on an outfit! it's comfortable and easy and flattering. bought this totally on ...

[thật=1 dự đoán=0 điểm=0.124]
   I wanted it to work I ordered a medium and it fit perfectly. i am anywhere from a size 6-8. 5'9, 155lbs.
sadly, i am returning this dress because the horizontal stripes make my hips look huge....

[thật=1 dự đoán=0 điểm=0.226]
   Sparkly & gold tones I didn't think from the photo online that the sweater would be quite so sparkly & gold! it has gold weaved in - so if the sparkly isn't your thing, beware. the sweater fit well although i didn't feel...

[thật=1 dự đoán=0 điểm=0.23]
   I loved this top so much i bought it in both colors. i have not seen anyone complain about this, but on me, the mock neck collapses in

**Quan sát.** Các trường hợp sai thường là bình luận **mâu thuẫn nội tại**: khen
kiểu dáng nhưng chê chất liệu, hoặc khen sản phẩm nhưng vẫn trả hàng vì sai size.

**Diễn giải.** Với biểu diễn túi-từ, một bình luận chứa cả từ tích cực lẫn tiêu
cực sẽ có hai nhóm tín hiệu triệt tiêu nhau. Mô hình không nắm được **cấu trúc
nhượng bộ** kiểu "đẹp *nhưng* không vừa" — mà chính vế sau mới quyết định nhãn.

**Ý nghĩa với học máy.** Đây là **giới hạn cố hữu của biểu diễn túi-từ**, không
phải của thuật toán. Muốn vượt qua thì phải đổi biểu diễn — sang embedding có
trật tự và mô hình đọc được ngữ cảnh — đúng như tensor $B \times T \times d$ minh
hoạ ở mục 12. Một lần nữa: **rào cản nằm ở biểu diễn, không nằm ở mô hình.**

## 21. Lựa chọn mô hình

| Tiêu chí | Nhận định |
|---|---|
| Hiệu năng dự báo | Xếp theo ROC-AUC và F1 lớp 0 trên tập test |
| Khả năng diễn giải | Logistic Regression cho hệ số **đọc được từng từ** — giá trị vận hành lớn |
| Chi phí tính toán | Mọi mô hình tuyến tính trên ma trận thưa đều huấn luyện trong vài giây |
| Độ bền | Linear SVM ổn định nhất trên dữ liệu thưa nhiều chiều |
| Ràng buộc triển khai | Phải lưu kèm từ điển TF-IDF; artifact lớn hơn hai ứng dụng kia |

Ở ứng dụng này, tiêu chí **diễn giải** có sức nặng lớn bất thường: bảng hệ số ở
mục 20 chính là sản phẩm "khám phá sở thích khách hàng" mà đề bài đòi hỏi. Một mô
hình hộp đen dù nhỉnh hơn vài phần nghìn ROC-AUC cũng không đánh đổi được điều đó.

In [25]:
display(test_tbl)
print(f"→ Mô hình được chọn để triển khai: {best_label}")
for m in ["ROC-AUC", "F1", "F1 (lớp 0)", "Accuracy"]:
    print(f"   {m:<12} = {test_tbl.loc[best_label, m]:.4f}")

,Biểu diễn,Accuracy,Precision,Recall,F1,F1 (lớp 0),ROC-AUC
Mô hình,,,,,,,
Logistic Regression (bảng + văn bản),Bảng + văn bản,0.8958,0.9530,0.9180,0.9352,0.7342,0.9431
Linear SVM (bảng + văn bản),Bảng + văn bản,0.8955,0.9509,0.9198,0.9351,0.7313,0.9414
Multinomial Naive Bayes (chỉ văn bản),Chỉ văn bản,0.8772,0.8855,0.9763,0.9287,0.5587,0.9346
Logistic Regression (chỉ bảng),Chỉ bảng,0.5389,0.8415,0.5383,0.6566,0.2984,0.5519
Decision Tree (chỉ bảng),Chỉ bảng,0.5197,0.8320,0.5182,0.6386,0.2843,0.5389
Random Forest (chỉ bảng),Chỉ bảng,0.6066,0.8244,0.6602,0.7332,0.2511,0.5304


→ Mô hình được chọn để triển khai: Logistic Regression (bảng + văn bản)
   ROC-AUC      = 0.9431
   F1           = 0.9352
   F1 (lớp 0)   = 0.7342
   Accuracy     = 0.8958


## 22. Lưu trữ mô hình

In [26]:
joblib.dump(pre_hybrid, MODEL_DIR / "preprocessor.joblib")
joblib.dump(pre_tabular, MODEL_DIR / "preprocessor_tabular.joblib")
joblib.dump(text_only, MODEL_DIR / "tfidf_text_only.joblib")
print("✓ preprocessor.joblib          (bảng + TF-IDF văn bản)")
print("✓ preprocessor_tabular.joblib  (chỉ bảng — dùng cho so sánh)")
print("✓ tfidf_text_only.joblib       (chỉ văn bản — dùng cho Naive Bayes)")
for key, spec in SPECS.items():
    # compress=3: mô hình rừng cây không nén chiếm hàng chục MB — quá lớn để đưa
    # vào kho mã nguồn. Nén zlib giảm khoảng 4 lần, gần như không ảnh hưởng tốc độ nạp.
    joblib.dump(trained[key], MODEL_DIR / f"{key}.joblib", compress=3)
    print(f"✓ {key}.joblib")

CAT_OPTIONS = {c: sorted(work[c].unique().tolist()) for c in CAT_FEATURES}

metadata = {
    "application": "customer_behavior",
    "task": "binary_classification",
    "random_seed": RANDOM_SEED,
    "numeric_features": NUM_FEATURES,
    "categorical_features": CAT_FEATURES,
    "text_column": TEXT_COL,
    "tabular_features": TAB_FEATURES,
    "categorical_options": CAT_OPTIONS,
    "target": "Recommended IND",
    "class_labels": {"0": "Không khuyến nghị sản phẩm", "1": "Khuyến nghị sản phẩm"},
    "excluded_features": {
        "Rating": "rò rỉ nhãn — quy tắc 'Rating >= 4' đã đạt accuracy "
                  + f"{acc_rule:.4f} mà không học gì",
        "Clothing ID": "định danh sản phẩm, không khái quát sang mã hàng mới"},
    "n_samples_raw": int(before),
    "n_samples_clean": int(after),
    "representation_shapes": {
        "tabular": list(Xtr_tab.shape),
        "hybrid": list(Xtr_hyb.shape),
        "text_only": list(Xtr_txt.shape)},
    "tensor_demo": {"B": int(B), "T": int(T), "d": int(D_EMB),
                    "vocab_size": int(V), "shape": list(E.shape)},
    "split": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "baseline_accuracy": round(float(BASE_ACC), 4),
    "best_model": best_id,
    "best_model_label": best_label,
    "best_model_repr": SPECS[best_id]["repr"],
    "model_repr": {k: s["repr"] for k, s in SPECS.items()},
    "model_labels": {k: s["label"] for k, s in SPECS.items()},
    "test_metrics": {
        [k for k, s in SPECS.items() if s["label"] == idx][0]: {
            m: float(test_tbl.loc[idx, m])
            for m in ["Accuracy", "Precision", "Recall", "F1", "F1 (lớp 0)", "ROC-AUC"]}
        for idx in test_tbl.index},
    "text_vs_tabular": summary.to_dict(),
    "top_positive_terms": {k: round(float(v), 4) for k, v in pos_terms.sort_values(ascending=False).items()},
    "top_negative_terms": {k: round(float(v), 4) for k, v in neg_terms.sort_values().items()},
    "confusion_matrix_best": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
}
(MODEL_DIR / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("✓ metadata.json")
print("\nArtifact đã lưu:")
for p in sorted(MODEL_DIR.iterdir()):
    print(f"   {p.name:<38} {p.stat().st_size/1024:>9.1f} KB")

✓ preprocessor.joblib          (bảng + TF-IDF văn bản)
✓ preprocessor_tabular.joblib  (chỉ bảng — dùng cho so sánh)
✓ tfidf_text_only.joblib       (chỉ văn bản — dùng cho Naive Bayes)
✓ logistic_regression_tabular.joblib
✓ decision_tree_tabular.joblib


✓ random_forest_tabular.joblib
✓ logistic_regression_hybrid.joblib
✓ linear_svm_hybrid.joblib
✓ multinomial_nb_text.joblib
✓ metadata.json

Artifact đã lưu:
   decision_tree_tabular.joblib                12.5 KB
   linear_svm_hybrid.joblib                   149.8 KB
   logistic_regression_hybrid.joblib          151.2 KB
   logistic_regression_tabular.joblib           0.8 KB
   metadata.json                                5.7 KB
   multinomial_nb_text.joblib                 502.4 KB
   preprocessor.joblib                        855.3 KB
   preprocessor_tabular.joblib                  4.2 KB
   random_forest_tabular.joblib              7346.4 KB
   tfidf_text_only.joblib                     792.6 KB


## 23. Kiểm thử suy luận

In [27]:
loaded_pre = joblib.load(MODEL_DIR / "preprocessor.joblib")
loaded_model = joblib.load(MODEL_DIR / f"{best_id}.joblib")
meta = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))

reviews = [
    {"Age": 34, "Positive Feedback Count": 3,
     "Review Text": "Absolutely love this dress! The fabric is soft, the fit is "
                    "perfect and I got so many compliments wearing it.",
     "Division Name": "General", "Department Name": "Dresses", "Class Name": "Dresses"},
    {"Age": 45, "Positive Feedback Count": 0,
     "Review Text": "Very disappointed. The material feels cheap and thin, it runs "
                    "two sizes too small and looks nothing like the picture. Returned it.",
     "Division Name": "General", "Department Name": "Tops", "Class Name": "Blouses"},
]

for i, r in enumerate(reviews, 1):
    text = r["Review Text"]
    frame = pd.DataFrame([{
        "Age": r["Age"],
        "Positive Feedback Count": r["Positive Feedback Count"],
        "word_count": len(text.split()),
        "Division Name": r["Division Name"],
        "Department Name": r["Department Name"],
        "Class Name": r["Class Name"],
        "full_text": text,
    }])
    vec = loaded_pre.transform(frame)          # transform, KHÔNG fit
    cls = int(loaded_model.predict(vec)[0])
    if hasattr(loaded_model, "predict_proba"):
        conf = float(loaded_model.predict_proba(vec)[0][cls])
    else:
        conf = float(1 / (1 + np.exp(-abs(loaded_model.decision_function(vec)[0]))))
    print(f"--- Đánh giá {i} ---")
    print(f'   "{text[:88]}..."')
    print(f"   Vectơ đầu vào: {vec.shape[1]:,} chiều (thưa)")
    print(f"   → Dự đoán: lớp {cls} — {meta['class_labels'][str(cls)]}")
    print(f"   → Độ tin cậy: {conf*100:.2f}%\n")

assert np.array_equal(loaded_model.predict(loaded_pre.transform(X_test)),
                      pred_best), "Artifact nạp lại KHÔNG khớp!"
print(f"✓ Artifact nạp lại cho kết quả trùng khớp trên {len(X_test):,} mẫu test.")
print("✓ Sẵn sàng cho REST API (api/REST_API.py).")

--- Đánh giá 1 ---
   "Absolutely love this dress! The fabric is soft, the fit is perfect and I got so many com..."
   Vectơ đầu vào: 20,034 chiều (thưa)
   → Dự đoán: lớp 1 — Khuyến nghị sản phẩm
   → Độ tin cậy: 99.30%

--- Đánh giá 2 ---
   "Very disappointed. The material feels cheap and thin, it runs two sizes too small and lo..."
   Vectơ đầu vào: 20,034 chiều (thưa)
   → Dự đoán: lớp 0 — Không khuyến nghị sản phẩm
   → Độ tin cậy: 99.89%



✓ Artifact nạp lại cho kết quả trùng khớp trên 3,396 mẫu test.
✓ Sẵn sàng cho REST API (api/REST_API.py).


## Tóm tắt Ứng dụng 3

| Hạng mục | Kết quả |
|---|---|
| Bài toán | Phân loại nhị phân |
| Dữ liệu | Women's E-Commerce Clothing Reviews — 23 486 × 11 |
| Vấn đề chất lượng chính | **Rò rỉ nhãn** qua cột `Rating`; mất cân bằng 4,5 : 1 |
| Biểu diễn | Bảng (~33 chiều) + TF-IDF (~20 000 chiều), ma trận thưa |
| Chuỗi biểu diễn văn bản | Bình luận → Token → Token ID → TF-IDF / $E \in \mathbb{R}^{B \times T \times d}$ |
| Số mô hình so sánh | 6 (3 chỉ-bảng, 2 lai, 1 chỉ-văn-bản) |
| Độ đo chính | F1 lớp thiểu số và ROC-AUC (accuracy vô dụng vì mất cân bằng) |
| Triển khai | Flask REST API + Web + giao diện Mobile |

Đóng góp riêng vào bài học chung: **cột dễ dùng nhất lại là cột phải vứt đi**
(`Rating` cho điểm số đẹp nhưng vô giá trị), còn **cột trông khó dùng nhất lại
mang gần như toàn bộ tín hiệu** (văn bản tự do). Chọn biểu diễn nào để đưa vào mô
hình là quyết định quan trọng hơn chọn thuật toán nào.